In [21]:
'''All c->w samples
Get highest activating neurons
Open mask for respective nerun
Calc alignment
Open expls for respective neuron
Clac concept dif %
At end: avg alignment and average concept dif %
If dif is high (close to 1) -> dif concepts entirely 
Alignment high: neurons behave the same so theyre learning dif ways to represent the same set → but those details are the wrong way
Alignment low: neurons explain vastly dif concepts and fire for very dif samples so the behavior of the neuron changes fully → wrong behavior
If dif is low (less than 50) -> dif combos 
High align: same behsvior even when misclassified
Low: wrong combos!'''
import pandas as pd
import numpy as np
import os
from collections import defaultdict
def all_correct_to_wrong(dense_cw, sparse_cw):
  
    
    return set(dense_cw['correct']) & set(sparse_cw['wrong'])
        

def find_highest_activating_neuron(samples_activations,model_finallayerweights, d):
    num_activ = (samples_activations>0).sum()
    contribution = torch.tensor(samples_activations).abs().squeeze(0) * model_finallayerweights.detach().cpu().abs()   # [3, 1024]

    
    total = contribution.sum(dim=1).argmax()   # [1024]

    most_impactful = contribution[total].argmax().item()
    return [most_impactful]
    

def iou(a,b):
    if isinstance(a, torch.Tensor):
        return (a&b).sum() / (a|b).sum()
    return len(a&b) / len(a|b) if len(a|b) > 0 else 0
ignore=0
def percent_sparse_in_dense(s,d):
    if len(d)==0 or len(s)==0: 
        return 1
    if len(d)==0 or len(s)==0: return 0
    return len(s&d)/len(d)

def concept_diff(expl_dense, expl_sparse):
    overlap = percent_sparse_in_dense(set(expl_dense), set(expl_sparse))
    return 1 - overlap

def get_mask(root,  cluster):
    print("root, ", root)
    return torch.load(os.path.join(root, f'Cluster{cluster}masks.pt'))

def get_subactiv(root, cluster):
    mask = get_mask(root,  cluster)
    dead=[]
    for n,i in enumerate(mask):
        if i.sum() < 500:
            dead.append(n)
    return dead
model='BERT'
method='wanda'
path_to_experiment=os.path.join("/workspace/CCE_NLI",model, 'exp', method, 'Run0.25_5')
dense_cw = pd.read_csv(os.path.join(path_to_experiment,  f'Prediction_CW_0_Pruning_Iter.csv')).set_index('Unnamed: 0').T
denseactivs = f"/workspace/CCE_NLI/{model}/activations/lottery_ticket/Run0.25_5/0_Pruning_Iter/final_layer_activations.pkl"
dense_expls,_=load_csv_data(os.path.join(f'/workspace/CCE_NLI/{model}/exp/lottery_ticket/Run0.25_5/Expls/', '0.0%Pruned/Cluster3IOUS1024N.csv'))
with open(denseactivs, 'rb') as f:
    dense_activations = pickle.load(f)
    
sparse_path_root = f"/workspace/CCE_NLI/{model}/exp/{method}/Run0.25_5"
dense_path_root = f"/workspace/CCE_NLI/{model}/exp/{method}/Run0.25_5"
dense_mask = get_mask(os.path.join(dense_path_root, 'Masks', '0.0%Pruned'), cluster=3)
start=1
dense_model_finallayerweights=torch.load(f'/workspace/CCE_NLI/{model}/models/lottery_ticket/Run0.25_5/0_Pruning_Iter/model_best.pth')['state_dict']['mlp.3.weight']
dense_dead=get_subactiv(os.path.join(dense_path_root, 'Masks', '0.0%Pruned'), cluster=3)
if '0.0%Pruned' in os.listdir(f"{sparse_path_root}/Expls"):
    start=1
for i, sparsity in enumerate(sorted(os.listdir(f"{sparse_path_root}/Expls"))):

    print(sparse_path_root, sparsity)
    #if sparsity != '25.0%Pruned': continue
    if '.ipynb' in sparsity or '0.0%Pruned' in sparsity: continue
    sparse_cw = pd.read_csv(os.path.join(path_to_experiment, f'Prediction_CW_{start}_Pruning_Iter.csv')).set_index('Unnamed: 0').T
    sparseactivs = f"/workspace/CCE_NLI/{model}/activations/{method}/Run0.25_5/{start}_Pruning_Iter/final_layer_activations.pkl"
    sparse_expls, _=load_csv_data(os.path.join(sparse_path_root, f'Expls/{sparsity}/Cluster3IOUS1024N.csv'))
    sparse_model_finallayerweights=torch.load(f'/workspace/CCE_NLI/{model}/models/{method}/Run0.25_5/{start}_Pruning_Iter/model_best.pth')['state_dict']['mlp.3.weight']
    
    with open(sparseactivs, 'rb') as f:
        sparse_activations = pickle.load(f)
        
    c_2_w = all_correct_to_wrong(dense_cw, sparse_cw)
    
    average_alignment = defaultdict(list)
    alignment_between_neurons= []
    average_concept_diff = defaultdict(int)
    sparse_mask = get_mask(os.path.join(sparse_path_root, 'Masks', sparsity), cluster=3)
    sparse_dead = get_subactiv(os.path.join(dense_path_root, 'Masks', sparsity), cluster=3)
    start += 1
    print(sparsity, len(c_2_w))
    died_neurons=0
    revived_neurons=0
    dense_highest_neuronsset=set()
    for c2w_sample in list(c_2_w):
 
        
        sparse_highest_neurons = find_highest_activating_neuron(sparse_activations[c2w_sample], sparse_model_finallayerweights, sparse_dead)
        dense_highest_neurons = find_highest_activating_neuron(dense_activations[c2w_sample],dense_model_finallayerweights, dense_dead )
        difference_in_concepts=0
        ignore = 0
        
        for sparse_highest_neuron, dense_highest_neuron in zip(sparse_highest_neurons, dense_highest_neurons):
            
            alignment_between_neurons=iou(sparse_mask[sparse_highest_neuron], dense_mask[dense_highest_neuron])
            average_alignment[c2w_sample].append(alignment_between_neurons.item())
            
            difference_in_concept = concept_diff(dense_expls[dense_highest_neuron], sparse_expls[sparse_highest_neuron]) #as percent
            #print(c2w_sample, dense_highest_neuron, dense_expls[dense_highest_neuron], sparse_highest_neuron, sparse_expls[sparse_highest_neuron], difference_in_concept, ignore)
            average_concept_diff[c2w_sample] += difference_in_concept
            
           
            
        
        average_alignment[c2w_sample] = sum(average_alignment[c2w_sample])/len(average_alignment[c2w_sample])
        
        #average_concept_diff[c2w_sample] /= (20-ignore)

    print(f"Average concept difference : ",   sum(list(average_concept_diff.values()))/len(average_concept_diff))
    print(f"Average alignment: ",  sum(list(average_alignment.values()))/len(average_alignment))
        
        

FileNotFoundError: [Errno 2] No such file or directory: '/workspace/CCE_NLI/BERT/exp/wanda/Run0.25_5/Prediction_CW_0_Pruning_Iter.csv'

In [16]:
contribution=torch.tensor([[i for i in range(5)] for i in range(3)])
total = contribution.sum(dim=0)   # [1024]
total

tensor([ 0,  3,  6,  9, 12])

In [5]:
import torch
import pickle
import numpy
import re
import pandas
import os

def get_indiv_concepts(formula) -> set:
    concepts = set()
    concps = re.findall(r'(?<!\bNOT\s)(?:\b(?:hyp|pre|oth):[^\s)]+)', formula)
    for c in concps:
        try:
            end_idx = c.index(')')
        except:
            end_idx = len(c)
        concepts.add(c[:end_idx])
    return concepts

def load_csv_data(filepath):
    """Load CSV and extract unit-concept mappings."""
    df = pd.read_csv(filepath)
    unit_concepts = defaultdict(set)
    raw=[]
    for _, row in df.iterrows():
        unit = row['unit']
        formula = row['best_name']
        concepts = get_indiv_concepts(formula)
        unit_concepts[unit].update(concepts)
        raw.extend(concepts)
    
    return unit_concepts, set(raw)

def build_binary_mask(neuron_mask, foundational_concept_list) -> torch.Tensor:
    num_neurons = len(neuron_mask)
    num_concepts = len(foundational_concept_list)

    # Step 1: Initialize tensor
    tensor = torch.zeros((num_neurons, num_concepts), dtype=torch.float32)

    # Step 2: Fill in ones
    for i, concepts in enumerate(neuron_mask.values()):
        for j, concept in enumerate(foundational_concept_list):
            if concept in concepts:
                tensor[i, j] = 1.0

    # Step 3: Compute row sums
    row_sums = tensor.sum(dim=1, keepdim=True)

    # Step 4: Normalize safely
    dist_tensor = torch.zeros_like(tensor)
    row_mask = (row_sums != 0).squeeze(1)  # True for rows with sum > 0
    dist_tensor[row_mask] = tensor[row_mask] #/ row_sums[row_mask]

    return dist_tensor



def get_neurons_for_cps(concepts, mapping):
    neurons = []
    for neuron, cps in mapping.items():
        for c in cps:
            if c in concepts:
                neurons.append(neuron)
                break
    return neurons

def get_all_cps_for_pi(folder):
    root_path = Path(folder)

    # Find all matching CSV files
    csv_pattern = 'Cluster*IOUS1024N.csv'
    csv_files = list(root_path.rglob(csv_pattern))
    
    concept_dict=defaultdict(lambda: defaultdict(set))
    s=set()
    for csv_file in csv_files:
        concepts = []
        csv_file = os.path.join(folder, csv_file)
        df = pd.read_csv(csv_file)
        for unit, formula in zip(df.unit, df.best_name):
            concept_dict[csv_file.split("/")[-2]][csv_file.split("/")[-1]].update(get_indiv_concepts(formula))
            s.update(get_indiv_concepts(formula))
     
      
    return concept_dict, s
